# Khai báo

In [2]:
import os, pandas as pd
import torch
import numpy as np
import pandas as pd
import os, json
import logging
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import gc
import pyarrow.parquet as pq
from collections import defaultdict

outputLabel = ["output_0","output_1","output_2","output_3","output_4","output_5"]
outputRegressLabel = ["lightning_0", "lightning_1", "lightning_2", "lightning_3", "lightning_4", "lightning_5"]
startPeriod      = -6
endPeriod        =  6
startPeriodTrain = -6
endPeriodTrain   =  0

bandName = ['B09B','B10B','B11B','B12B','B14B','B16B','I2B','I4B','IRB','WVB', 'NDVI','Dem_value']
# bandKep = ['IRB-I2B', 'B09B-B14B', 'WVB-B14B', 'B11B-B14B', 'I2B-B14B', 'WVB-IRB', 'B11B-IRB', 'B14B-I2B', 'B11B-B12B', 'WVB-B10B']

# bandName = bandName + bandKep

bandType = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriod, endPeriod)
    for band in bandName
]

bandTypeTrain = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriodTrain, endPeriodTrain)
    for band in bandName
]

# Train - Val - Test
# Chưa chuẩn hóa 
# Sử dụng mật độ trục Y, Trục X dùng train P2 và P98

# 10 BANDS HIMA trước này
# BANDS NDVI cũng có nhưng mà không chuẩn hóa
# BANDS DEM

# Train

In [3]:
TRAIN_TEST_DIR = '/sdd/Dubaoset/src/Thang/DataMB/TrainProcess'

In [3]:
# listFile = ['/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_0.parquet']
listFileVal = [VAL_PATH]

In [11]:
band_list = ", ".join([f'MIN("{b}") AS "min_{b}", AVG("{b}") AS "mean_{b}", MEDIAN("{b}") AS "median_{b}", MAX("{b}") AS "max_{b}", PERCENTILE_CONT(0.02) WITHIN GROUP (ORDER BY "{b}") AS "p2_{b}", PERCENTILE_CONT(0.98) WITHIN GROUP (ORDER BY "{b}") AS "p98_{b}"' for b in bandType])

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import json

# ── 1. Thiết lập kết nối ──
con = duckdb.connect()
con.execute("SET memory_limit='30GB'")
con.execute("SET threads=8")

# Giả định listFile và bandTypeTrain đã được định nghĩa
file_list_str = ", ".join([f"'{f}'" for f in listFile])

# Nhóm các cột theo band gốc
band_groups = defaultdict(list)
for col in bandType:
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

# Biến để lưu trữ thông số chuẩn dùng cho tập Test sau này
train_standard_scales = {}
all_stats = []
hist_results = {}

for band_name, cols in band_groups.items():
    print(f"Đang xử lý {band_name}...")
    
    # Tạo luồng dữ liệu gom các timestamp lại
    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    # Bước 2.1: Tính Stats (Dùng để xác định khung thước đo)
    stats_query = f"""
    SELECT
        '{band_name}' AS band,
        COUNT(*) AS total_rows,
        MIN(val) AS min_val,
        AVG(val) AS mean_val,
        APPROX_QUANTILE(val, 0.5) AS median_val,
        MAX(val) AS max_val,
        APPROX_QUANTILE(val, 0.02) AS p2,
        APPROX_QUANTILE(val, 0.98) AS p98
    FROM ({union_parts})
    """
    s_df = con.execute(stats_query).df()
    stats = s_df.iloc[0]
    all_stats.append(s_df)

    # LƯU BIẾN CHO TRAIN/TEST: Lưu lại p2, p98 làm chuẩn
    train_standard_scales[band_name] = {
        'p2': float(stats['p2']),
        'p98': float(stats['p98']),
        'mean': float(stats['mean_val'])
    }
    
    # Bước 2.2: Thiết lập Bins dựa trên khung đã chốt
    p2, p98 = stats['p2'], stats['p98']
    val_range = p98 - p2
    
    if band_name == 'NDVI': n_bins = 100
    elif band_name == 'Dem_value': n_bins = 150
    else: n_bins = 80 
        
    # Bước 2.3: Tính Histogram dạng MẬT ĐỘ (DENSITY)
    # Công thức: (COUNT(*) / Tổng số dòng) * 100 để ra %
    hist_query = f"""
    SELECT
        floor((val - {p2}) / (NULLIF({val_range}, 0)) * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / {stats['total_rows']} as density_pct
    FROM ({union_parts})
    WHERE val BETWEEN {p2} AND {p98}
    GROUP BY bin_idx ORDER BY bin_idx
    """
    h_df = con.execute(hist_query).df()
    hist_results[band_name] = {'df': h_df, 'stats': stats, 'n_bins': n_bins}

# ── 3. Vẽ đồ thị với Mật độ ──
fig, axes = plt.subplots(len(band_groups), 1, figsize=(12, 5 * len(band_groups)))
if len(band_groups) == 1: axes = [axes]

for ax, band_name in zip(axes, band_groups.keys()):
    data = hist_results[band_name]
    h_df, s, n_bins = data['df'], data['stats'], data['n_bins']
    
    p2, p98 = s['p2'], s['p98']
    bin_width = (p98 - p2) / n_bins
    bin_centers = p2 + (h_df['bin_idx'] + 0.5) * bin_width
    
    # Vẽ Bar chart với trục Y là % mật độ
    ax.bar(bin_centers, h_df['density_pct'], width=bin_width * 0.9, 
           color="steelblue", alpha=0.7, label="Train Distribution")
    
    ax.axvline(s['mean_val'], color="green", lw=2, label=f"Mean: {s['mean_val']:.2f}")
    ax.set_ylabel("Mật độ (%)")
    ax.set_xlabel("Giá trị")
    
    # Ép trục X theo đúng P2-P98 của Train
    ax.set_xlim(p2, p98)
    
    ax.set_title(f"Band: {band_name} | Train Density (Total: {s['total_rows']:,} pts)", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()
plt.savefig("/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_non_normalize_full12.png", dpi=150)
print("Đã lưu thông số chuẩn vào train_scales.json và ảnh vào train_density_report.png")

In [ ]:
# Val_non_normalize
# Test_non_normalize cùng thang đo với Train khi chưa chuẩn hóa nha
import re
import duckdb
import matplotlib.pyplot as plt
con = duckdb.connect()
con.execute("SET memory_limit='20GB'")
con.execute("SET threads=3")

try:
    with open('train_scales.json', 'r') as f:
        train_standard_scales = json.load(f)
    print("--- Đã load thành công thước đo chuẩn từ Train ---")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file train_scales.json. Hãy chạy bước Train trước!")
    exit()

file_list_str = ", ".join([f"'{f}'" for f in listFileVal])

band_groups = defaultdict(list)
for col in bandType:
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

test_hist_results = {}

# ── 2. Tính toán Histogram cho Test dựa trên thước đo Train ──
for band_name, cols in band_groups.items():
    if band_name not in train_standard_scales:
        continue
        
    print(f"Đang xử lý tập Test cho Band: {band_name}...")
    
    p2 = train_standard_scales[band_name]['p2']
    p98 = train_standard_scales[band_name]['p98']
    mean_tr = train_standard_scales[band_name]['mean']
    val_range = p98 - p2
    
    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    if band_name == 'NDVI': n_bins = 100
    elif band_name == 'Dem_value': n_bins = 150
    else: n_bins = 80 

    # SỬA LẠI SQL:
    # 1. Dùng GREATEST và LEAST để "clip" dữ liệu vào khoảng P2-P98 (Thấy được Drift)
    # 2. Không scale về 0-1, giữ nguyên giá trị thật để tính bin
    # 3. Mẫu số chia cho TỔNG SỐ DÒNG (raw_data)
    hist_query = f"""
    WITH raw_data AS (
        SELECT val FROM ({union_parts})
    ),
    clipped_data AS (
        SELECT 
            least(greatest(val, {p2}), {p98}) AS val_clipped 
        FROM raw_data
    )
    SELECT
        floor((val_clipped - {p2}) / NULLIF({val_range}, 0) * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM raw_data) as density_pct
    FROM clipped_data
    GROUP BY bin_idx 
    ORDER BY bin_idx
    """
    
    h_df = con.execute(hist_query).df()
    test_actual_stats = con.execute(f"SELECT AVG(val) as m FROM ({union_parts})").df().iloc[0]
    
    test_hist_results[band_name] = {
        'df': h_df, 
        'test_mean': test_actual_stats['m'],
        'n_bins': n_bins,
        'p2': p2,
        'p98': p98,
        'mean_tr': mean_tr
    }

# ── 3. Vẽ đồ thị so sánh (SỬA LẠI TRỤC X) ──
fig, axes = plt.subplots(len(test_hist_results), 1, figsize=(12, 5 * len(test_hist_results)))
if len(test_hist_results) == 1: axes = [axes]

for ax, band_name in zip(axes, test_hist_results.keys()):
    res = test_hist_results[band_name]
    h_df = res['df']
    
    p2_tr = res['p2']
    p98_tr = res['p98']
    
    # Tính lại bin_centers theo GIÁ TRỊ THẬT (không phải 0-1)
    bin_width = (p98_tr - p2_tr) / res['n_bins']
    bin_centers = p2_tr + (h_df['bin_idx'] + 0.5) * bin_width
    
    # Vẽ cột
    ax.bar(bin_centers, h_df['density_pct'], width=bin_width * 0.9, 
           color="orange", alpha=0.7, label="Val/Test Distribution")
    
    # Vẽ đường Mean gốc (Giá trị thật, không scale)
    ax.axvline(res['mean_tr'], color="blue", lw=2, label=f"Train Mean: {res['mean_tr']:.2f}")
    ax.axvline(res['test_mean'], color="red", lw=2, linestyle="--", label=f"Test Mean: {res['test_mean']:.2f}")
    
    # Giới hạn trục X đúng bằng P2-P98 của Train
    ax.set_xlim(p2_tr, p98_tr)
    ax.set_title(f"Comparison Band: {band_name} (Val vs Train Scale)", fontweight='bold')
    ax.set_xlabel("Giá trị (Original Scale)")
    ax.set_ylabel("Mật độ (%)")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/val_non_normalize.png", dpi=150)
print("Đã vẽ xong ảnh val_non_normalize.png")

## Sau chuẩn hóa
# Train

In [ ]:
# Với normalize thì ấy cho HIMA và DEM thôi 
# Khai báo lại

In [2]:
import os, pandas as pd
import torch
import numpy as np
import pandas as pd
import os, json
import logging
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import gc
import pyarrow.parquet as pq
from collections import defaultdict

outputLabel = ["output_0","output_1","output_2","output_3","output_4","output_5"]
outputRegressLabel = ["lightning_0", "lightning_1", "lightning_2", "lightning_3", "lightning_4", "lightning_5"]
startPeriod      = -6
endPeriod        =  6
startPeriodTrain = -6
endPeriodTrain   =  0
# DEM cho max
# NDVI không cần
bandName = ['B09B','B10B','B11B','B12B','B14B','B16B','I2B','I4B','IRB','WVB']
# bandKep = ['IRB-I2B', 'B09B-B14B', 'WVB-B14B', 'B11B-B14B', 'I2B-B14B', 'WVB-IRB', 'B11B-IRB', 'B14B-I2B', 'B11B-B12B', 'WVB-B10B']

# bandName = bandName + bandKep

bandType = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriod, endPeriod)
    for band in bandName
]

bandTypeTrain = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriodTrain, endPeriodTrain)
    for band in bandName
]

In [3]:
bandType

['B09B_t-6',
 'B10B_t-6',
 'B11B_t-6',
 'B12B_t-6',
 'B14B_t-6',
 'B16B_t-6',
 'I2B_t-6',
 'I4B_t-6',
 'IRB_t-6',
 'WVB_t-6',
 'B09B_t-5',
 'B10B_t-5',
 'B11B_t-5',
 'B12B_t-5',
 'B14B_t-5',
 'B16B_t-5',
 'I2B_t-5',
 'I4B_t-5',
 'IRB_t-5',
 'WVB_t-5',
 'B09B_t-4',
 'B10B_t-4',
 'B11B_t-4',
 'B12B_t-4',
 'B14B_t-4',
 'B16B_t-4',
 'I2B_t-4',
 'I4B_t-4',
 'IRB_t-4',
 'WVB_t-4',
 'B09B_t-3',
 'B10B_t-3',
 'B11B_t-3',
 'B12B_t-3',
 'B14B_t-3',
 'B16B_t-3',
 'I2B_t-3',
 'I4B_t-3',
 'IRB_t-3',
 'WVB_t-3',
 'B09B_t-2',
 'B10B_t-2',
 'B11B_t-2',
 'B12B_t-2',
 'B14B_t-2',
 'B16B_t-2',
 'I2B_t-2',
 'I4B_t-2',
 'IRB_t-2',
 'WVB_t-2',
 'B09B_t-1',
 'B10B_t-1',
 'B11B_t-1',
 'B12B_t-1',
 'B14B_t-1',
 'B16B_t-1',
 'I2B_t-1',
 'I4B_t-1',
 'IRB_t-1',
 'WVB_t-1',
 'B09B_t+0',
 'B10B_t+0',
 'B11B_t+0',
 'B12B_t+0',
 'B14B_t+0',
 'B16B_t+0',
 'I2B_t+0',
 'I4B_t+0',
 'IRB_t+0',
 'WVB_t+0',
 'B09B_t+1',
 'B10B_t+1',
 'B11B_t+1',
 'B12B_t+1',
 'B14B_t+1',
 'B16B_t+1',
 'I2B_t+1',
 'I4B_t+1',
 'IRB_t+1',
 'WV

In [10]:
listFile = [os.path.join(TRAIN_DATA_DIR, f) for f in os.listdir(TRAIN_DATA_DIR) if f.endswith(".parquet")]
listFile

['/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_0.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_10.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_15.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_20.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_25.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_30.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_35.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_40.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_5.parquet']

In [8]:
listFile = [listFile[0]]

In [ ]:
## SAu chuẩn hóa
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import json

# ── 1. Thiết lập kết nối ──
con = duckdb.connect()
con.execute("SET memory_limit='20GB'")
con.execute("SET threads=16")

# Giả định listFile và bandTypeTrain đã được định nghĩa
file_list_str = ", ".join([f"'{f}'" for f in listFile])

# Nhóm các cột theo band gốc
band_groups = defaultdict(list)
for col in bandType:
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

# Biến để lưu trữ thông số chuẩn dùng cho tập Test sau này
train_standard_scales = {}
all_stats = []
hist_results = {}

# ── 2. Tính toán Stats & Histogram (Mật độ) ──
for band_name, cols in band_groups.items():
    print(f"Đang xử lý {band_name}...")
    
    # Tạo luồng dữ liệu gom các timestamp lại
    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    # Bước 2.1: Tính Stats (Dùng để xác định khung thước đo)
    stats_query = f"""
    SELECT
        '{band_name}' AS band,
        COUNT(*) AS total_rows,
        MIN(val) AS min_val,
        AVG(val) AS mean_val,
        APPROX_QUANTILE(val, 0.5) AS median_val,
        MAX(val) AS max_val,
        APPROX_QUANTILE(val, 0.02) AS p2,
        APPROX_QUANTILE(val, 0.98) AS p98
    FROM ({union_parts})
    """
    s_df = con.execute(stats_query).df()
    stats = s_df.iloc[0]
    all_stats.append(s_df)

    p2_orig = float(stats['p2'])
    p98_orig = float(stats['p98'])
    val_range = p98_orig - p2_orig

    def scale_value(x):
        if val_range == 0:
            return 0.0
        return (float(x) - p2_orig) / val_range

    # LƯU BIẾN CHO TRAIN/TEST: Lưu thành 2 cụm Rõ Ràng (Gốc và Đã chuẩn hóa)
    train_standard_scales[band_name] = {
        'original': {
            'min': float(stats['min_val']),
            'max': float(stats['max_val']),
            'p2': p2_orig,
            'p98': p98_orig,
            'mean': float(stats['mean_val']),
            'median': float(stats['median_val'])
        },
        'scaled': {
            'min': scale_value(stats['min_val']),
            'max': scale_value(stats['max_val']),
            'p2': scale_value(p2_orig),         # Cái này in ra chắc chắn sẽ là 0.0
            'p98': scale_value(p98_orig),       # Cái này in ra chắc chắn sẽ là 1.0
            'mean': scale_value(stats['mean_val']),
            'median': scale_value(stats['median_val'])
        }
    }
    
    # Bước 2.2: Thiết lập Bins dựa trên khung đã chốt
    p2, p98 = p2_orig, p98_orig
    val_range = p98 - p2
    
    if band_name == 'NDVI': n_bins = 100
    elif band_name == 'Dem_value': n_bins = 150
    else: n_bins = 80 
        
    # Bước 2.3: Tính Histogram dạng MẬT ĐỘ (DENSITY)
    # Công thức: (COUNT(*) / Tổng số dòng) * 100 để ra %
    hist_query = f"""
    WITH scaled_data AS (
        SELECT 
            -- Công thức Min-Max Scaling đưa val về khoảng 0-1
            (val - {p2}) / NULLIF({val_range}, 0) AS val_scaled
        FROM ({union_parts})
        WHERE val BETWEEN {p2} AND {p98}
    )
    SELECT
        -- Chia 80 bin trong khoảng từ 0 đến 1
        floor(val_scaled * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM scaled_data) as density_pct
    FROM scaled_data
    GROUP BY bin_idx 
    ORDER BY bin_idx
    """
    h_df = con.execute(hist_query).df()
    hist_results[band_name] = {'df': h_df, 'stats': stats, 'n_bins': n_bins}

# Lưu thông số ra file JSON để dùng cho tập Test
with open("/sdd/Dubaoset/src/Thang/dataStatistic/output_plots/train_normalization_stats_full12.json", "w") as f:
    json.dump(train_standard_scales, f, indent=4)
print("Đã lưu toàn bộ thông số chuẩn hóa ra file train_normalization_stats_full12.json!")

# ── 3. Vẽ đồ thị với Mật độ ──
fig, axes = plt.subplots(len(band_groups), 1, figsize=(12, 5 * len(band_groups)))
if len(band_groups) == 1: axes = [axes]

for ax, band_name in zip(axes, band_groups.keys()):
    data = hist_results[band_name]
    h_df, s, n_bins = data['df'], data['stats'], data['n_bins']
    
    # Lấy thông số để chuẩn hóa đường Mean
    p2, p98 = s['p2'], s['p98']
    val_range = p98 - p2
    
    # VÌ DATA TRONG h_df BÂY GIỜ LÀ TỪ 0 ĐẾN 1:
    # 1. Chiều rộng bin trên thang 0-1
    bin_width_scaled = 1.0 / n_bins
    # 2. Tâm bin trên thang 0-1
    bin_centers_scaled = (h_df['bin_idx'] + 0.5) * bin_width_scaled

    # Vẽ Bar chart (Trục X bây giờ là 0-1)
    ax.bar(bin_centers_scaled, h_df['density_pct'], width=bin_width_scaled * 0.9, 
           color="seagreen", alpha=0.7, label="Train (Min-Max Scaled)")
    
    # 3. Chuẩn hóa Mean về thang 0-1 để vẽ đường thẳng đứng
    scaled_mean = (s['mean_val'] - p2) / val_range if val_range != 0 else 0
    ax.axvline(scaled_mean, color="red", lw=2, linestyle="--", 
               label=f"Scaled Mean: {scaled_mean:.2f}")
    
    # Thiết lập trục
    ax.set_xlim(0, 1) # CỐ ĐỊNH TRỤC X TỪ 0 ĐẾN 1
    ax.set_ylabel("Mật độ (%)")
    ax.set_xlabel("Giá trị chuẩn hóa (0=P2, 1=P98)")
    
    # Chú thích thêm giá trị vật lý ở tiêu đề để dễ hình dung
    ax.set_title(f"Band: {band_name} | Scaled 0-1 (Original P2:{p2:.1f}, P98:{p98:.1f})", 
                 fontsize=12, fontweight='bold')
    
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig("/sdd/Dubaoset/src/Thang/dataStatistic/output_plots/train_normalized_full12.png", dpi=150)
plt.show()

In [1]:
listFile = ['/sdd/Dubaoset/src/Thang/ChooseData/total_5_clean.parquet']
listFile

['/sdd/Dubaoset/src/Thang/ChooseData/total_5_clean.parquet']

In [4]:
# Val and test
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import json

# ── 1. Thiết lập kết nối ──
con = duckdb.connect()
con.execute("SET memory_limit='30GB'")
con.execute("SET threads=16")

# Giả định listFile chứa danh sách các file Parquet của tập VAL (hoặc TEST)
file_list_str = ", ".join([f"'{f}'" for f in listFile])

# Nhóm các cột theo band gốc
band_groups = defaultdict(list)
for col in bandType:
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

# ── 2. LOAD THƯỚC ĐO CỦA TẬP TRAIN ──
TRAIN_STATS_PATH = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_normalization_stats.json"
with open(TRAIN_STATS_PATH, "r") as f:
    train_stats = json.load(f)

val_tracking_stats = {} # Biến mới để lưu log thông số của tập Val/Test
hist_results = {}

# ── 3. TÍNH TOÁN, LƯU FILE VÀ CHUẨN HÓA MẬT ĐỘ ──
for band_name, cols in band_groups.items():
    print(f"Đang xử lý {band_name} trên tập Validation...")
    
    if band_name not in train_stats:
        print(f"⚠️ Bỏ qua {band_name} vì không có thông số Train!")
        continue

    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    # --- BƯỚC A: Tính thông số thực tế của tập Val ---
    stats_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(val) AS min_val,
        AVG(val) AS mean_val,
        APPROX_QUANTILE(val, 0.5) AS median_val,
        MAX(val) AS max_val,
        APPROX_QUANTILE(val, 0.02) AS p2,
        APPROX_QUANTILE(val, 0.98) AS p98
    FROM ({union_parts})
    """
    v_df = con.execute(stats_query).df()
    v_stats = v_df.iloc[0]
    
    # Rút thước đo của Train ra để chuẩn bị tính toán
    p2_train = train_stats[band_name]['original']['p2']
    p98_train = train_stats[band_name]['original']['p98']
    val_range_train = p98_train - p2_train

    # Hàm tính giá trị Scaled (chưa bị clip) để xem Val lệch bao nhiêu so với Train
    def scale_val_value(x):
        if val_range_train == 0: return 0.0
        return (float(x) - p2_train) / val_range_train

    # LƯU DICTIONARY ĐẦY ĐỦ (Gốc và Đã Scale)
    val_tracking_stats[band_name] = {
        'total_rows': int(v_stats['total_rows']),
        'used_train_scale': {
            'p2_train': p2_train,
            'p98_train': p98_train
        },
        'val_original_stats': {
            'min': float(v_stats['min_val']),
            'max': float(v_stats['max_val']),
            'p2': float(v_stats['p2']),
            'p98': float(v_stats['p98']),
            'mean': float(v_stats['mean_val']),
            'median': float(v_stats['median_val'])
        },
        'val_scaled_stats_before_clip': {
            'min': scale_val_value(v_stats['min_val']),
            'max': scale_val_value(v_stats['max_val']),
            'p2': scale_val_value(v_stats['p2']),
            'p98': scale_val_value(v_stats['p98']),
            'mean': scale_val_value(v_stats['mean_val']),
            'median': scale_val_value(v_stats['median_val'])
        }
    }

    # --- BƯỚC B: CHUẨN HÓA BẰNG THÔNG SỐ TRAIN VÀ TÍNH HISTOGRAM ---
    if band_name == 'NDVI': n_bins = 100
    elif band_name == 'Dem_value': n_bins = 150
    else: n_bins = 80 
        
    hist_query = f"""
    WITH scaled_data AS (
        SELECT 
            -- Ép tràn: Dùng P2 và P98 của Train
            GREATEST(0.0, LEAST(1.0, (val - {p2_train}) / NULLIF({val_range_train}, 0))) AS val_scaled
        FROM ({union_parts})
    )
    SELECT
        floor(val_scaled * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM scaled_data) as density_pct
    FROM scaled_data
    GROUP BY bin_idx 
    ORDER BY bin_idx
    """
    h_df = con.execute(hist_query).df()
    
    hist_results[band_name] = {
        'df': h_df, 
        'val_mean': float(v_stats['mean_val']), 
        'p2_train': p2_train, 
        'p98_train': p98_train, 
        'val_range_train': val_range_train,
        'n_bins': n_bins
    }

# ── 4. GHI LOG RA FILE JSON ──
VAL_STATS_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/NewData/val_tracking_stats.json"
with open(VAL_STATS_OUTPUT, "w") as f:
    json.dump(val_tracking_stats, f, indent=4)
print(f"✅ Đã ghi thông số thống kê của tập Val ra file: {VAL_STATS_OUTPUT}")

# ── 5. VẼ ĐỒ THỊ VÀ LƯU ẢNH ──
fig, axes = plt.subplots(len(band_groups), 1, figsize=(12, 5 * len(band_groups)))
if len(band_groups) == 1: axes = [axes]

for ax, band_name in zip(axes, band_groups.keys()):
    if band_name not in hist_results:
        continue
        
    data = hist_results[band_name]
    h_df = data['df']
    n_bins = data['n_bins']
    p2_t, p98_t, range_t = data['p2_train'], data['p98_train'], data['val_range_train']
    
    bin_width_scaled = 1.0 / n_bins
    bin_centers_scaled = (h_df['bin_idx'] + 0.5) * bin_width_scaled

    ax.bar(bin_centers_scaled, h_df['density_pct'], width=bin_width_scaled * 0.9, 
           color="darkorange", alpha=0.7, label="Validation Data")
    
    # Vẽ đường Mean của chính tập Val (nhưng đã bị scale theo thước đo Train)
    scaled_val_mean = (data['val_mean'] - p2_t) / range_t if range_t != 0 else 0
    ax.axvline(scaled_val_mean, color="red", lw=2, linestyle="--", 
               label=f"Val Mean (Scaled): {scaled_val_mean:.2f}")
    
    ax.set_xlim(-0.05, 1.05) 
    ax.set_ylabel("Mật độ (%)")
    ax.set_xlabel("Giá trị chuẩn hóa")
    ax.set_title(f"Band: {band_name} | Val Data Scaled by Train (Train P2:{p2_t:.1f}, Train P98:{p98_t:.1f})", 
                 fontsize=12, fontweight='bold')
    
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
PLOT_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/NewData/new_train_normalized_by_old_train.png"
plt.savefig(PLOT_OUTPUT, dpi=150)
print(f"✅ Đã lưu ảnh plot đồ thị ra file: {PLOT_OUTPUT}")
plt.close()

Đang xử lý B09B trên tập Validation...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Đang xử lý B10B trên tập Validation...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Đang xử lý WVB trên tập Validation...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Đã ghi thông số thống kê của tập Val ra file: /sdd/Dubaoset/src/Thang/dataStatistic/NewData/val_tracking_stats.json
✅ Đã lưu ảnh plot đồ thị ra file: /sdd/Dubaoset/src/Thang/dataStatistic/NewData/new_train_normalized_by_old_train.png


# DEM

In [28]:
# Code DEM max
import os, pandas as pd
import torch
import numpy as np
import pandas as pd
import os, json
import logging
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import gc
import pyarrow.parquet as pq
from collections import defaultdict

outputLabel = ["output_0","output_1","output_2","output_3","output_4","output_5"]
outputRegressLabel = ["lightning_0", "lightning_1", "lightning_2", "lightning_3", "lightning_4", "lightning_5"]
startPeriod      = -6
endPeriod        =  6
startPeriodTrain = -6
endPeriodTrain   =  0
# DEM cho max
# NDVI không cần
bandName = ['Dem_value']
# bandKep = ['IRB-I2B', 'B09B-B14B', 'WVB-B14B', 'B11B-B14B', 'I2B-B14B', 'WVB-IRB', 'B11B-IRB', 'B14B-I2B', 'B11B-B12B', 'WVB-B10B']

# bandName = bandName + bandKep

bandType = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriod, endPeriod)
    for band in bandName
]

bandTypeTrain = [
    f"{band}_t{('-' + str(-i)) if i < 0 else ('+' + str(i))}" if (band != "Dem_value" and band != "DEMIsLand") else band
    for i in range(startPeriodTrain, endPeriodTrain)
    for band in bandName
]

In [36]:
listFile = [os.path.join(TRAIN_DATA_DIR, f) for f in os.listdir(TRAIN_DATA_DIR) if f.endswith(".parquet")]
listFile

['/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_0.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_10.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_15.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_20.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_25.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_30.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_35.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_40.parquet',
 '/sdd/Dubaoset/src/Phong/Model/data/trainClean/merged_data_part_5.parquet']

In [37]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import json

# ── 1. Thiết lập kết nối ──
con = duckdb.connect()
con.execute("SET memory_limit='30GB'")
con.execute("SET threads=16")

# Giả định listFile chứa danh sách các file Parquet của tập TRAIN
file_list_str = ", ".join([f"'{f}'" for f in listFile])

# Nhóm các cột theo band gốc
band_groups = defaultdict(list)
for col in bandTypeTrain:
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

train_max_stats = {} 
hist_results = {}

# ── 2. TÍNH TOÁN STATS, HISTOGRAM VÀ CHUẨN HÓA MAX ──
for band_name, cols in band_groups.items():
    print(f"Đang xử lý {band_name} trên tập Train (Max Normalization)...")
    
    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    # --- BƯỚC A: Tìm thông số gốc, đặc biệt là MAX ABSOLUTE ---
    stats_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(val) AS min_val,
        AVG(val) AS mean_val,
        APPROX_QUANTILE(val, 0.5) AS median_val,
        MAX(val) AS max_val,
        MAX(ABS(val)) AS max_abs_val  -- TÌM GIÁ TRỊ TUYỆT ĐỐI LỚN NHẤT ĐỂ CHIA
    FROM ({union_parts})
    """
    s_df = con.execute(stats_query).df()
    stats = s_df.iloc[0]
    
    max_abs_orig = float(stats['max_abs_val'])
    
    # Hàm tính nhanh giá trị sau khi scale Max
    def scale_by_max(x):
        if max_abs_orig == 0: return 0.0
        return float(x) / max_abs_orig

    # Lưu cả giá trị gốc và giá trị sau khi chuẩn hóa vào Dict
    train_max_stats[band_name] = {
        'total_rows': int(stats['total_rows']),
        'original': {
            'min': float(stats['min_val']),
            'max': float(stats['max_val']),
            'max_abs': max_abs_orig,
            'mean': float(stats['mean_val']),
            'median': float(stats['median_val'])
        },
        'scaled': {
            'min': scale_by_max(stats['min_val']),
            'max': scale_by_max(stats['max_val']),
            'max_abs': 1.0,  # Cái này chia xong chắc chắn bằng 1.0
            'mean': scale_by_max(stats['mean_val']),
            'median': scale_by_max(stats['median_val'])
        }
    }

    # --- BƯỚC B: TÍNH HISTOGRAM TRÊN DẢI [-1, 1] ---
    n_bins = 150 if band_name == 'Dem_value' else 100
        
    hist_query = f"""
    WITH scaled_data AS (
        SELECT 
            -- Chuẩn hóa: Chia thẳng cho Max Abs
            val / NULLIF({max_abs_orig}, 0) AS val_scaled
        FROM ({union_parts})
    )
    SELECT
        -- Đưa dải [-1, 1] về [0, 1] bằng cách cộng 1 rồi chia 2, sau đó nhân với n_bins
        floor(((val_scaled + 1.0) / 2.0) * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM scaled_data) as density_pct
    FROM scaled_data
    GROUP BY bin_idx 
    ORDER BY bin_idx
    """
    h_df = con.execute(hist_query).df()
    
    hist_results[band_name] = {
        'df': h_df, 
        'stats': train_max_stats[band_name], 
        'n_bins': n_bins
    }

# ── 3. GHI LOG RA FILE JSON ──
TRAIN_STATS_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_max_normalization_stats_dem.json"
with open(TRAIN_STATS_OUTPUT, "w") as f:
    json.dump(train_max_stats, f, indent=4)
print(f"✅ Đã ghi thước đo Max của tập Train ra file: {TRAIN_STATS_OUTPUT}")

# ── 4. VẼ ĐỒ THỊ VÀ LƯU ẢNH ──
fig, axes = plt.subplots(len(band_groups), 1, figsize=(12, 5 * len(band_groups)))
if len(band_groups) == 1: axes = [axes]

for ax, band_name in zip(axes, band_groups.keys()):
    if band_name not in hist_results: continue
        
    data = hist_results[band_name]
    h_df = data['df']
    n_bins = data['n_bins']
    max_abs_t = data['stats']['original']['max_abs']
    scaled_mean = data['stats']['scaled']['mean']
    
    # Tính toán tọa độ X cho các cột Bin (trên dải -1 đến 1)
    # Vì công thức bin gộp dải [-1, 1] lại thành n_bins, nên chiều rộng bin trên trục thực tế là 2.0 / n_bins
    bin_width = 2.0 / n_bins
    
    # Phục hồi giá trị tâm của bin từ bin_idx
    # Công thức đảo: val_scaled = (bin_idx / (n_bins-1)) * 2.0 - 1.0 (cộng thêm 0.5 bin_width để lấy tâm)
    h_df['bin_center'] = (h_df['bin_idx'] / (n_bins - 1)) * 2.0 - 1.0 + (bin_width / 2.0)

    ax.bar(h_df['bin_center'], h_df['density_pct'], width=bin_width * 0.9, 
           color="royalblue", alpha=0.7, label="Train Data (Max Scaled)")
    
    # Vẽ đường Mean (đã chuẩn hóa)
    ax.axvline(scaled_mean, color="red", lw=2, linestyle="--", 
               label=f"Scaled Mean: {scaled_mean:.2f}")
    
    # THIẾT LẬP TRỤC X CHO DẢI TỪ -1 ĐẾN 1 (Thêm lề 0.05 cho đẹp)
    ax.set_xlim(-1.05, 1.05) 
    ax.set_ylabel("Mật độ (%)")
    ax.set_xlabel("Giá trị chuẩn hóa (Chia cho Max)")
    ax.set_title(f"Band: {band_name} | Max Scaled (Max_Abs: {max_abs_t:.1f})", 
                 fontsize=12, fontweight='bold')
    
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
PLOT_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_max_normalized_dem.png"
plt.savefig(PLOT_OUTPUT, dpi=150)
print(f"✅ Đã lưu ảnh plot đồ thị ra file: {PLOT_OUTPUT}")
plt.close()

Đang xử lý Dem_value trên tập Train (Max Normalization)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Đã ghi thước đo Max của tập Train ra file: /sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_max_normalization_stats_dem.json
✅ Đã lưu ảnh plot đồ thị ra file: /sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_max_normalized_dem.png


In [ ]:
# Val và test

In [38]:
listFile = [VAL_PATH]
listFile

['/sdd/Dubaoset/src/Phong/Model/data/unknown/clean_eval_data_6.parquet']

In [39]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re
import json

# ── 1. Thiết lập kết nối ──
con = duckdb.connect()
con.execute("SET memory_limit='30GB'")
con.execute("SET threads=16")

# Giả định listFile chứa danh sách các file Parquet của tập VAL (hoặc TEST)
file_list_str = ", ".join([f"'{f}'" for f in listFile])

# Nhóm các cột theo band gốc
band_groups = defaultdict(list)
for col in bandType: 
    if col in ("Dem_value", "DEMIsLand"):
        band_groups[col].append(col)
        continue
    m = re.match(r"^(.+)_t[+\-]\d+$", col)
    if m:
        band_groups[m.group(1)].append(col)

# ── 2. LOAD THƯỚC ĐO CỦA TẬP TRAIN ──
TRAIN_STATS_PATH = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/train_max_normalization_stats_dem.json"
with open(TRAIN_STATS_PATH, "r") as f:
    train_stats = json.load(f)

val_max_tracking_stats = {} 
hist_results = {}

# ── 3. TÍNH TOÁN, LƯU FILE VÀ CHUẨN HÓA MẬT ĐỘ ──
for band_name, cols in band_groups.items():
    print(f"Đang xử lý {band_name} trên tập Validation (Max Normalization)...")
    
    if band_name not in train_stats:
        print(f"⚠️ Bỏ qua {band_name} vì không có thông số Train!")
        continue

    union_parts = "\n    UNION ALL\n    ".join([
        f'SELECT "{c}" AS val FROM read_parquet([{file_list_str}]) WHERE "{c}" IS NOT NULL'
        for c in cols
    ])
    
    # --- BƯỚC A: Tính thông số thực tế của tập Val ---
    stats_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(val) AS min_val,
        AVG(val) AS mean_val,
        APPROX_QUANTILE(val, 0.5) AS median_val,
        MAX(val) AS max_val,
        MAX(ABS(val)) AS max_abs_val
    FROM ({union_parts})
    """
    v_df = con.execute(stats_query).df()
    v_stats = v_df.iloc[0]
    
    # Rút thước đo MAX ABS của Train ra để chuẩn bị tính toán
    max_abs_train = train_stats[band_name]['original']['max_abs']

    # Hàm tính giá trị Scaled (chưa bị clip) để xem Val lệch bao nhiêu so với Train
    def scale_val_by_train_max(x):
        if max_abs_train == 0: return 0.0
        return float(x) / max_abs_train

    # LƯU DICTIONARY ĐẦY ĐỦ (Gốc và Đã Scale Trước Khi Clip)
    val_max_tracking_stats[band_name] = {
        'total_rows': int(v_stats['total_rows']),
        'used_train_scale': {
            'max_abs_train': max_abs_train
        },
        'val_original_stats': {
            'min': float(v_stats['min_val']),
            'max': float(v_stats['max_val']),
            'max_abs': float(v_stats['max_abs_val']),
            'mean': float(v_stats['mean_val']),
            'median': float(v_stats['median_val'])
        },
        'val_scaled_stats_before_clip': {
            'min': scale_val_by_train_max(v_stats['min_val']),
            'max': scale_val_by_train_max(v_stats['max_val']),
            'max_abs': scale_val_by_train_max(v_stats['max_abs_val']),
            'mean': scale_val_by_train_max(v_stats['mean_val']),
            'median': scale_val_by_train_max(v_stats['median_val'])
        }
    }

    # --- BƯỚC B: CHUẨN HÓA, ÉP TRÀN VÀ TÍNH HISTOGRAM TRÊN DẢI [-1, 1] ---
    n_bins = 150 if band_name == 'Dem_value' else 100
        
    hist_query = f"""
    WITH scaled_data AS (
        SELECT 
            -- Ép tràn: Gò cứng dữ liệu trong khoảng -1.0 đến 1.0 an toàn cho Model
            GREATEST(-1.0, LEAST(1.0, val / NULLIF({max_abs_train}, 0))) AS val_scaled
        FROM ({union_parts})
    )
    SELECT
        -- Đưa dải [-1, 1] về [0, 1] bằng cách cộng 1 rồi chia 2, sau đó nhân với n_bins
        floor(((val_scaled + 1.0) / 2.0) * ({n_bins} - 1)) as bin_idx,
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM scaled_data) as density_pct
    FROM scaled_data
    GROUP BY bin_idx 
    ORDER BY bin_idx
    """
    h_df = con.execute(hist_query).df()
    
    hist_results[band_name] = {
        'df': h_df, 
        'val_scaled_mean_before_clip': scale_val_by_train_max(v_stats['mean_val']), 
        'max_abs_train': max_abs_train,
        'n_bins': n_bins
    }

# ── 4. GHI LOG RA FILE JSON ──
VAL_STATS_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/val_max_tracking_stats_dem.json"
with open(VAL_STATS_OUTPUT, "w") as f:
    json.dump(val_max_tracking_stats, f, indent=4)
print(f"✅ Đã ghi thông số thống kê của tập Val ra file: {VAL_STATS_OUTPUT}")

# ── 5. VẼ ĐỒ THỊ VÀ LƯU ẢNH ──
fig, axes = plt.subplots(len(band_groups), 1, figsize=(12, 5 * len(band_groups)))
if len(band_groups) == 1: axes = [axes]

for ax, band_name in zip(axes, band_groups.keys()):
    if band_name not in hist_results: continue
        
    data = hist_results[band_name]
    h_df = data['df']
    n_bins = data['n_bins']
    max_abs_t = data['max_abs_train']
    scaled_val_mean = data['val_scaled_mean_before_clip']
    
    # Tính toán tọa độ X cho các cột Bin (trên dải -1 đến 1)
    bin_width = 2.0 / n_bins
    h_df['bin_center'] = (h_df['bin_idx'] / (n_bins - 1)) * 2.0 - 1.0 + (bin_width / 2.0)

    # Đổi màu thành darkorange để phân biệt với tập Train
    ax.bar(h_df['bin_center'], h_df['density_pct'], width=bin_width * 0.9, 
           color="darkorange", alpha=0.7, label="Val Data (Max Scaled & Clipped)")
    
    # Vẽ đường Mean của tập Val (đã scale)
    ax.axvline(scaled_val_mean, color="red", lw=2, linestyle="--", 
               label=f"Val Mean (Scaled): {scaled_val_mean:.2f}")
    
    # THIẾT LẬP TRỤC X CHO DẢI TỪ -1 ĐẾN 1 (Thêm lề 0.05)
    ax.set_xlim(-1.05, 1.05) 
    ax.set_ylabel("Mật độ (%)")
    ax.set_xlabel("Giá trị chuẩn hóa (Chia cho Train Max_Abs)")
    ax.set_title(f"Band: {band_name} | Val Data Scaled by Train (Train Max_Abs: {max_abs_t:.1f})", 
                 fontsize=12, fontweight='bold')
    
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
PLOT_OUTPUT = "/sdd/Dubaoset/src/Thang/dataStatistic/Teacher/val_max_normalized_dem.png"
plt.savefig(PLOT_OUTPUT, dpi=150)
print(f"✅ Đã lưu ảnh plot đồ thị ra file: {PLOT_OUTPUT}")
plt.close()

Đang xử lý Dem_value trên tập Validation (Max Normalization)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Đã ghi thông số thống kê của tập Val ra file: /sdd/Dubaoset/src/Thang/dataStatistic/Teacher/val_max_tracking_stats_dem.json
✅ Đã lưu ảnh plot đồ thị ra file: /sdd/Dubaoset/src/Thang/dataStatistic/Teacher/val_max_normalized_dem.png
